# Quantum Annealing QUBOs for PSD Anomaly and RACH Detection
This mini-notebook builds QUBOs for:
1. PSD multi-class hypothesis selection (benign/fake/tone).
2. RACH root-delay selection.
We validate them by classical solve and export the coefficient dictionaries.

In [ ]:
import numpy as np, numpy.fft as fft, json, itertools, math
import matplotlib.pyplot as plt
print('Env OK')

## RF helpers

In [ ]:
def gen_ofdm_like(N=1024, subcarriers=(60,120,200), amps=(1.0,0.8,0.6), noise=0.05):
    t=np.arange(N); x=np.zeros(N,dtype=complex)
    for f,a in zip(subcarriers,amps): x += a*np.exp(1j*2*np.pi*f*t/N)
    x += noise*(np.random.randn(N)+1j*np.random.randn(N)); return x
def shift_sync(x, shift_bins=6): X=fft.fft(x); X=np.roll(X,shift_bins); return fft.ifft(X)
def add_tone(x, bin_idx=150, amp=3.0): N=len(x); n=np.arange(N); return x+amp*np.exp(1j*2*np.pi*bin_idx*n/N)
def psd_feature(x, n_bins=16): P=np.abs(fft.fft(x))**2; k=len(P)//n_bins; P=P[:k*n_bins].reshape(n_bins,k).mean(axis=1); return (P/(P.max()+1e-12)).astype(float)
def zadoff_chu(u,Nzc): n=np.arange(Nzc); return np.exp(-1j*np.pi*u*n*(n+1)/Nzc)
def rach_burst(zc, delay=0, snr_db=12.0): N=len(zc); x=np.zeros(N,dtype=complex); x[delay:]=zc[:N-delay]; snr=10**(snr_db/10); p=np.mean(np.abs(x)**2)+1e-12; npwr=p/snr; noise=np.sqrt(npwr/2)*(np.random.randn(N)+1j*np.random.randn(N)); return x+noise
def corr_refnorm(rx, ref, n_bins=16, ref_scale=None): r=np.correlate(rx,ref,mode='same'); c=np.abs(r); 
    if ref_scale is None: rr=np.correlate(ref,ref,mode='same'); ref_scale=float(np.max(np.abs(rr)));
    c=c/(ref_scale+1e-12); k=len(c)//n_bins; return c[:k*n_bins].reshape(n_bins,k).mean(axis=1).astype(float)


## PSD QUBO

In [ ]:
np.random.seed(0); B=16
xb=gen_ofdm_like(); xf=shift_sync(gen_ofdm_like(),6); xt=add_tone(gen_ofdm_like(),150,3.0)
mu={'benign': psd_feature(xb,B), 'fake': psd_feature(xf,B), 'tone': psd_feature(xt,B)}
def E_class(x_feat): return {c: float(np.sum((x_feat-mu[c])**2)) for c in mu}
def build_psd_qubo(Ec, A=5.0):
    classes=list(Ec.keys()); n=len(classes)
    Q=np.zeros((n,n)); cvec=np.zeros(n)
    for i,c in enumerate(classes): cvec[i]=Ec[c]-A
    for i in range(n):
        for j in range(i+1,n): Q[i,j]+=2*A
    return classes,Q,cvec
def solve_onehot(Q,cvec):
    n=len(cvec); bestE=1e9; best=None
    for i in range(n):
        z=np.zeros(n); z[i]=1
        E=float(z@Q@z + cvec@z)
        if E<bestE: bestE=E; best=z
    return best,bestE
x_new = np.random.choice([gen_ofdm_like(), shift_sync(gen_ofdm_like(),6), add_tone(gen_ofdm_like(),150,3.0)])
f_new = psd_feature(x_new,B); Ec=E_class(f_new)
classes,Q,cvec=build_psd_qubo(Ec,A=5.0); z*,E*=None, None

In [ ]:
# Evaluate solution (separate cell to avoid star-syntax issues)
zstar, Estar = solve_onehot(Q,cvec)
pred = classes[int(np.argmax(zstar))]
print('Classes:', classes)
print('Costs:', {k: round(v,3) for k,v in Ec.items()})
print('QUBO prediction:', pred)

## RACH QUBO

In [ ]:
np.random.seed(1)
Nzc=839; roots=[167,173,181]; zc={u: zadoff_chu(u,Nzc) for u in roots}
ref_scale={u: float(np.max(np.abs(np.correlate(zc[u],zc[u],mode='same')))) for u in roots}
delays=list(range(8,17))
def C_matrix(rx):
    C=np.zeros((len(roots),len(delays)))
    for i,u in enumerate(roots):
        for j,d in enumerate(delays):
            feat=corr_refnorm(rx, zc[u], n_bins=16, ref_scale=ref_scale[u])
            C[i,j] = -float(np.max(feat))
    return C
rx = rach_burst(zc[167], delay=12, snr_db=12.0)
C = C_matrix(rx)
def build_rach_qubo(C, A=10.0, B=10.0):
    K,L=C.shape; n=K+L; Q=np.zeros((n,n)); c=np.zeros(n)
    for k in range(K):
        for l in range(L): Q[k,K+l]+=0.5*C[k,l]
    for k in range(K): c[k]+=-A
    for k1 in range(K):
        for k2 in range(k1+1,K): Q[k1,k2]+=2*A
    for l in range(L): c[K+l]+=-B
    for l1 in range(L):
        for l2 in range(l1+1,L): Q[K+l1,K+l2]+=2*B
    return Q,c
Q_rach, c_rach = build_rach_qubo(C,10.0,10.0)
def solve_rach(Q,c):
    K=len(roots); L=len(delays); n=K+L; bestE=1e9; best=None
    for ki in range(K):
        for lj in range(L):
            z=np.zeros(n); z[ki]=1; z[K+lj]=1
            E=float(z@Q@z + c@z)
            if E<bestE: bestE=E; best=z
    return best,bestE
zstar2,Estar2=solve_rach(Q_rach,c_rach)
k_hat=int(np.argmax(zstar2[:len(roots)])); l_hat=int(np.argmax(zstar2[len(roots):]))
print('Predicted root/delay:', roots[k_hat], delays[l_hat])

## Export dictionaries

In [ ]:
def export_qubo(Q,c,labels=None,path='/mnt/data/qubo.json'):
    Q=np.array(Q); c=np.array(c)
    obj={'Q_upper': {(int(i),int(j)): float(Q[i,j]) for i in range(Q.shape[0]) for j in range(i,Q.shape[1]) if abs(Q[i,j])>1e-12},
         'c': {int(i): float(ci) for i,ci in enumerate(c) if abs(ci)>1e-12},
         'labels': labels or {}}
    import json; fpath=path
    with open(fpath,'w') as f: json.dump(obj,f,indent=2)
    return fpath
psd_export = export_qubo(np.zeros((len(classes),len(classes))), cvec, labels={'classes': classes}, path='/mnt/data/psd_onehot_linear.json')
rach_export = export_qubo(Q_rach, c_rach, labels={'roots': roots, 'delays': delays}, path='/mnt/data/rach_root_delay_qubo.json')
print('Exported:', psd_export)
print('Exported:', rach_export)

## Ocean-Compatible Exports and D-Wave Sampler Stubs
This section adds utilities to:
- convert our QUBO `(Q, c)` into Ocean **BQM/QUBO** formats,
- optionally sample on a D-Wave system (hybrid or QPU) if the `dwave-system` stack is available,
- otherwise fall back to classical validation.

**Note:** These cells are safe to run without Ocean installed; they export JSON that Ocean can read.

In [ ]:
def qubo_upper_from_Q_c(Q_mat, c_vec):
    """Return an Ocean-style upper-tri QUBO dict: {(i,j): coeff}, including diagonal via c_vec."""
    import numpy as _np
    Q_mat = _np.array(Q_mat, dtype=float)
    c_vec = _np.array(c_vec, dtype=float)
    n = len(c_vec)
    Q = {}
    # add off-diagonal from Q_mat
    for i in range(n):
        for j in range(i+1, n):
            if abs(Q_mat[i,j]) > 1e-12:
                Q[(i,j)] = float(Q_mat[i,j])
    # add diagonal using c_vec and any existing diag in Q_mat
    for i in range(n):
        diag = float(c_vec[i] + (Q_mat[i,i] if i < Q_mat.shape[0] else 0.0))
        if abs(diag) > 1e-12:
            Q[(i,i)] = diag
    return Q

def export_qubo_ocean_json(Q_mat, c_vec, labels=None, path='/mnt/data/qubo_ocean.json'):
    Q = qubo_upper_from_Q_c(Q_mat, c_vec)
    obj = {
        'type': 'QUBO',
        'Q': {f'{i},{j}': float(v) for (i,j),v in Q.items()},
        'labels': labels or {}
    }
    with open(path,'w') as f:
        json.dump(obj, f, indent=2)
    return path

def export_bqm_if_dimod(Q_mat, c_vec, path='/mnt/data/bqm_serialized.json'):
    try:
        import dimod
        Q = qubo_upper_from_Q_c(Q_mat, c_vec)
        bqm = dimod.BinaryQuadraticModel.from_qubo(Q, offset=0.0)
        # Use built-in serialization if available
        try:
            from dimod.serialization.json import dump as dimod_dump
            with open(path, 'w') as f:
                dimod_dump(bqm, f)
        except Exception:
            # Fallback: a simple dict with linear/quadratic
            data = {
                'vartype': 'BINARY',
                'linear': {str(v): float(bqm.get_linear(v)) for v in bqm.variables},
                'quadratic': {f'{u},{v}': float(bqm.get_quadratic(u,v)) for u,v in bqm.quadratic},
                'offset': float(bqm.offset)
            }
            with open(path, 'w') as f:
                json.dump(data, f, indent=2)
        return path
    except Exception as e:
        print('dimod not available; skipped BQM export:', e)
        return None

In [ ]:
# --- PSD instance export (uses the most recent Ec/classes from above) ---
try:
    classes, Q_psd, c_psd = build_psd_qubo(Ec, A=5.0)
    ocean_psd = export_qubo_ocean_json(Q_psd, c_psd, labels={'classes': classes}, path='/mnt/data/psd_qubo_ocean.json')
    bqm_psd = export_bqm_if_dimod(Q_psd, c_psd, path='/mnt/data/psd_bqm.json')
    print('PSD exports:', ocean_psd, bqm_psd)
except Exception as e:
    print('PSD export failed:', e)

In [ ]:
# --- RACH export ---
try:
    ocean_rach = export_qubo_ocean_json(Q_rach, c_rach, labels={'roots': roots, 'delays': delays}, path='/mnt/data/rach_qubo_ocean.json')
    bqm_rach = export_bqm_if_dimod(Q_rach, c_rach, path='/mnt/data/rach_bqm.json')
    print('RACH exports:', ocean_rach, bqm_rach)
except Exception as e:
    print('RACH export failed:', e)

### D-Wave sampler examples (optional)
The following cells are stubs: they attempt to import the `dwave-system` stack and, if present,
submit either a QUBO dict or a BQM to a **hybrid** sampler. Set `RUN_ON_DWAVE=True` and ensure your
environment has valid D-Wave credentials (e.g., via `dwave setup`).

In [ ]:
RUN_ON_DWAVE = False  # set to True if your environment is configured with dwave-system
if RUN_ON_DWAVE:
    try:
        import dimod
        from dwave.system import LeapHybridSampler
        # Example with PSD QUBO
        Q_qubo = qubo_upper_from_Q_c(Q_psd, c_psd)
        sampler = LeapHybridSampler()
        # Hybrid accepts BQM; build BQM from QUBO
        bqm = dimod.BinaryQuadraticModel.from_qubo(Q_qubo)
        res = sampler.sample(bqm, label='PSD one-hot demo', time_limit=3)
        print('Hybrid sample (PSD):', res.first.sample, 'energy:', res.first.energy)
        # Example with RACH
        Q_qubo_r = qubo_upper_from_Q_c(Q_rach, c_rach)
        bqm_r = dimod.BinaryQuadraticModel.from_qubo(Q_qubo_r)
        res_r = sampler.sample(bqm_r, label='RACH root-delay demo', time_limit=3)
        print('Hybrid sample (RACH):', res_r.first.sample, 'energy:', res_r.first.energy)
    except Exception as e:
        print('D-Wave hybrid sampling failed:', e)
else:
    print('Set RUN_ON_DWAVE=True to attempt hybrid sampling (requires dwave-system).')

In [ ]:
RUN_ON_QPU = False
if RUN_ON_QPU:
    try:
        import dimod
        from dwave.system import DWaveSampler, EmbeddingComposite
        # Keep problem very small for QPU demo
        Q_small = {(0,0):-1.0, (1,1):-1.0, (0,1):2.0}
        bqm_small = dimod.BinaryQuadraticModel.from_qubo(Q_small)
        sampler = EmbeddingComposite(DWaveSampler())
        sampleset = sampler.sample(bqm_small, num_reads=2000, label='Tiny demo QPU')
        print(sampleset.first.sample, sampleset.first.energy)
    except Exception as e:
        print('QPU sampling failed:', e)
else:
    print('Set RUN_ON_QPU=True to attempt QPU sampling (requires dwave-system & QPU access).')